<a href="https://colab.research.google.com/github/prometheus404/NLP_proj/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initiazlization

In [1]:
%pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

/bin/zsh: /home/prometheus/anaconda3/envs/llama-env/lib/libncursesw.so.6: no version information available (required by /bin/zsh)
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Params
VERBOSE = True
CHOSEN = 'llama'
FILE_NAMES = ['ticket_to_ride', 'dominion', 'catan', 'power_grid_recharged']
IT = 5
BASE_URL = 'https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/flawed_texts/'
OVERWRITE = ['']
try:
    #DRIVE
    from google.colab import drive
    drive.mount('/content/drive',force_remount=True)
    BASE_FOLDER = 'drive/MyDrive/NLP_proj/error_detection/'
    N_GPU_LAYERS = -1
except:
    #LOCAL
    BASE_FOLDER = 'error_detection/'
    N_GPU_LAYERS = 20

In [3]:
from llama_cpp import Llama, llama_free, llama_free_model
from tqdm import tqdm
#from transformers import AutoTokenizer, pipeline, BitsAndBytesConfig
import requests
from collections import defaultdict
import json
import torch
import os

# Load the model
models = {
    'llama': {'repo_id':"bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
              'filename':"Meta-Llama-3.1-8B-Instruct-Q6_K.gguf",
              'temperature': 0.7,
              'n_ctx': 32768,
              'chat_format': "llama-3"
              },
    'qwen': {'repo_id':"bartowski/Qwen2.5-7B-Instruct-GGUF",
             'filename': "Qwen2.5-7B-Instruct-Q6_K.gguf",
             'temperature': 0.6,
             'n_ctx': 32768,
             'chat_format': "qwen"},
    'gemma': {'repo_id':"bartowski/google_gemma-3n-E4B-it-GGUF",
                'filename':"google_gemma-3n-E4B-it-Q6_K.gguf",
                'temperature': 0.7,
                'n_ctx': 32768,
                'chat_format': None
             },
}
model = Llama.from_pretrained(repo_id=models[CHOSEN]['repo_id'], # repository name
                            filename=models[CHOSEN]['filename'], # model file
                            n_gpu_layers=N_GPU_LAYERS, # use all GPU layers
                            n_ctx=models[CHOSEN]['n_ctx'], # context size
                            flash_attn=True, # use flash attention
                            chat_format=models[CHOSEN]['chat_format'], # chat format
                            verbose=VERBOSE,
                            force_download=True,
                            enable_thinking=True)

def generate_message(sys_prompt, usr_prompt):
    return [
        {
                "role": "system",
                "content": sys_prompt,
            },
            {
                "role": "user",
                "content": usr_prompt,
            },
    ]

/home/prometheus/anaconda3/envs/llama-env/lib/python3.9/site-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    yes
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA GeForce GTX 1070, compute capability 6.1, VMM: yes
llama_model_load_from_file_impl: using device CUDA0 (NVIDIA GeForce GTX 1070) - 7647 MiB free
llama_model_loader: loaded meta data with 33 key-value pairs and 292 tensors from /home/prometheus/.cache/huggingface/hub/models--bartowski--Meta-Llama-3.1-8B-Instruct-GGUF/snapshots/bf5b95e96dac0462e2a09145ec66cae9a3f12067/./Meta-Llama-3.1-8B-Instruct-Q6_K.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:  

# Error detection

 Each rulebook is edited by inserting errors that are increasingly difficult to spot:
 - level 0: **original**    -> unaltered rulebook
 - level 1: **missing**     -> an entire paragraph of the rulebook describing some core mechanic is missing
- level 2: **unsolvable**   -> a rule directly contradicts another one
- level 3: **incoherent**   -> a combination of rules hardlocks the game
- level 4: **gamebreaking** -> a coherent but obviously unbalanced mechanic

In [ ]:
prompt = """You are an expert game board player.
Examine the rulebook provided by the user.
Proceed with a chain of thought:

- Scan the text linearly and note any statements that conflict with earlier ones.
- For each of the game mechanic check if it is explained.
- Check whether any mechanic could halt the game or give a player an overwhelming advantage.

Report the **most impactful** problem you discover, quoting the relevant line and summarizing its impact.
If nothing stands out, reply “The rules appear consistent.”
"""



output_dict = {'lvl'+str(lvl): {str(it): '' for it in range(IT)} for lvl in range(5)}
to_do = [g for g in FILE_NAMES if f'{CHOSEN}_{g}.json' not in os.listdir(BASE_FOLDER)
                                 or f'{CHOSEN}_{g}.json' in OVERWRITE]

if to_do == []:
    print('Nothing to do')

for g in to_do:
    for it,lvl in tqdm([(it,lvl) for it in range(IT) for lvl in range(5)]):
        rulebook = requests.get(BASE_URL+'lvl'+str(lvl)+'/'+g+'.txt').text #rulebook is cached so should not be a problem
        name = g.replace('_',' ')
        out = model.create_chat_completion(generate_message(prompt, "Here is the full rulebook of the game {name}:\n"+rulebook),
                                           temperature=models[CHOSEN]['temperature'])['choices'][0]['message']['content']
        output_dict['lvl'+str(lvl)][str(it)] = out
        if(VERBOSE):
            print(f'{g} lvl{lvl}, iteration: {it}\n---------\n{out}')


    with open(f'{BASE_FOLDER}{CHOSEN}_{g}.json','w') as f:
        json.dump(dict(output_dict),f)

  0%|                                                    | 0/25 [00:00<?, ?it/s]llama_perf_context_print:        load time =    5487.07 ms
llama_perf_context_print: prompt eval time =    5484.64 ms /  1965 tokens (    2.79 ms per token,   358.27 tokens per second)
llama_perf_context_print:        eval time =   70025.12 ms /   327 runs   (  214.14 ms per token,     4.67 tokens per second)
llama_perf_context_print:       total time =   75884.71 ms /  2292 tokens
llama_perf_context_print:    graphs reused =        325
  4%|█▊                                          | 1/25 [01:16<30:39, 76.65s/it]

ticket_to_ride lvl0, iteration: 0
---------
After scanning the rulebook, I've identified a potential problem that could give a player an overwhelming advantage.

**Problem:** The game mechanic for drawing train cards allows a player to draw 2 cards, either from the face-up cards next to the board or from the top of the deck (a blind draw). However, if a player draws a locomotive card as their first card, they cannot draw a second card. This is stated in the rule: "If you want to draw a face up locomotive, it must be the first train card you draw this turn and you cannot then draw a second card."

**Impact:** This rule creates an advantage for the player who draws a locomotive card first, as they can only draw one card for the rest of their turn. In contrast, if a player draws a non-locomotive card first, they can still draw a second card. This means that players who draw locomotives early in the game can potentially limit their opponents' ability to draw cards and make strategic decisi

Llama.generate: 1025 prefix-match hit, remaining 516 prompt tokens to eval
llama_perf_context_print:        load time =    5487.07 ms
llama_perf_context_print: prompt eval time =    1704.74 ms /   516 tokens (    3.30 ms per token,   302.69 tokens per second)
llama_perf_context_print:        eval time =   61251.94 ms /   317 runs   (  193.22 ms per token,     5.18 tokens per second)
llama_perf_context_print:       total time =   63314.62 ms /   833 tokens
llama_perf_context_print:    graphs reused =        315
  8%|███▌                                        | 2/25 [02:20<26:25, 68.93s/it]

ticket_to_ride lvl1, iteration: 0
---------
After examining the rulebook, I have identified a potential problem that could give one player an overwhelming advantage.

The issue is with the "Draw Train Cards" mechanic, specifically the rule that states:

"If, at any time, 3 of the 5 face up train cards are locomotives, all 5 cards are immediately discarded, and 5 new cards are turned face up to replace them."

This means that if a player draws a locomotive as their first train card, and then the top 5 face-up cards are discarded and replaced, the player who drew the locomotive gets to draw 5 new face-up cards, including potentially multiple locomotives. This could result in a player getting a large number of locomotives in a single turn, allowing them to create a very long path and potentially winning the game.

This mechanic could be seen as problematic because it allows a player to "streak" and get a large number of locomotives in a single turn, which could give them a significant adv

Llama.generate: 682 prefix-match hit, remaining 1287 prompt tokens to eval
llama_perf_context_print:        load time =    5487.07 ms
llama_perf_context_print: prompt eval time =    3712.29 ms /  1287 tokens (    2.88 ms per token,   346.69 tokens per second)
llama_perf_context_print:        eval time =   82285.71 ms /   386 runs   (  213.18 ms per token,     4.69 tokens per second)
llama_perf_context_print:       total time =   86451.36 ms /  1673 tokens
llama_perf_context_print:    graphs reused =        383
 12%|█████▎                                      | 3/25 [03:46<28:14, 77.02s/it]

...

power_grid_recharged lvl1, iteration: 2
---------
After examining the rulebook, I have identified a problem that could potentially halt the game or give a player an overwhelming advantage.

**Problem:** The rules do not specify what happens when a player's power plants are unable to store the required resources to produce electricity. This is mentioned in the rules for Phase 3: Buy Resources, where it says "Players may not buy resources that their current power plants cannot use." However, the rules do not specify what happens to the player's turn or the game's progress when this situation arises.

**Impact:** This problem could potentially lead to a situation where a player is unable to buy resources to produce electricity, causing a game-halting situation. If the rules are not clear on how to handle this situation, it could lead to disputes and confusion among players.

**Relevant line:** "Players may not buy resources that their current power plants cannot use."

**Additional notes:

llama_perf_context_print:        load time =    5487.07 ms
llama_perf_context_print: prompt eval time =   14604.41 ms /  3844 tokens (    3.80 ms per token,   263.21 tokens per second)
llama_perf_context_print:        eval time =  215148.25 ms /   500 runs   (  430.30 ms per token,     2.32 tokens per second)
llama_perf_context_print:       total time =  230386.59 ms /  4344 tokens
llama_perf_context_print:    graphs reused =        497
 52%|█████████████████████▊                    | 13/25 [36:01<35:34, 177.84s/it]Llama.generate: 1149 prefix-match hit, remaining 6337 prompt tokens to eval


power_grid_recharged lvl2, iteration: 2
---------
After thoroughly examining the rulebook, I have identified a potential problem with the game mechanics.

**Most impactful problem:**

The problem lies in the way resources are resupplied during each phase. Specifically, in Phase 5 (Bureaucracy), the resource refill summary card dictates the amount of each resource type to be resupplied, based on the number of players and the current Step of the game. However, the game does not account for the fact that players may have stored large numbers of tokens on their power plants, which can lead to a situation where there are not enough resource tokens of a particular type left in the supply.

**Quoting the relevant line:**

"Resource resupply for 5 players:

|         | Step 1 | Step 2 | Step 3 |
|---------|--------|--------|--------|
| Coal    | 5      | 7      | 5      |
| Oil     | 4      | 5      | 6      |
| Garbage | 3      | 3      | 5      |
| Uranium | 2      | 3      | 2      |

Examp

llama_perf_context_print:        load time =    5487.07 ms
llama_perf_context_print: prompt eval time =   22102.54 ms /  6337 tokens (    3.49 ms per token,   286.71 tokens per second)
llama_perf_context_print:        eval time =  109432.07 ms /   259 runs   (  422.52 ms per token,     2.37 tokens per second)
llama_perf_context_print:       total time =  131821.79 ms /  6596 tokens
llama_perf_context_print:    graphs reused =        257
 56%|███████████████████████▌                  | 14/25 [38:13<30:03, 164.00s/it]Llama.generate: 1149 prefix-match hit, remaining 6349 prompt tokens to eval


power_grid_recharged lvl3, iteration: 2
---------
After carefully examining the rulebook, I have found a problem that stands out as potentially impactful. Here's the issue:

**Problem:** In Step 3, the rulebook states that "there is no future market in Step 3." However, it also states that the power plant market will be updated at the beginning of Phase 5 (Bureaucracy) in Step 3, which implies that the future market is still present, but its behavior is different.

**Quote:** "During Step 3, remove the smallest numbered power plant in the current market from the game and draw a replacement from the power plant stack instead."

**Impact:** This discrepancy could lead to confusion and inconsistent gameplay. The rulebook should clearly state whether the future market is present in Step 3 or not. If it is present, the rules for updating the power plant market should be consistent with the rest of the game. If it is not present, the rules for updating the market should be clearly defined.

